In [ ]:
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,confusion_matrix,classification_report,recall_score,f1_score

def read_file(path) -> pd.DataFrame:
  if path:
    df=pd.read_csv(path)
    df.drop_duplicates(inplace=True)
    print("missing value in dataset\n",df.isna().sum())
    print("sample of data\n",df.head())
    return df
  else:
   raise FileNotFoundError("file not found")


In [45]:
def preprocessing(df) -> pd.DataFrame:
  df["age"]=df["age"].fillna(df.groupby("cholesterol")['age'].transform('median')).fillna(df["age"].median())

  df["age_bins"]=pd.cut(df["age"] , bins=[0,10,20,35,50,60,70,80,100])
  df["glucose_bins"]=pd.cut(df["glucose_level"],bins=[70,80,90,100,120,140])
  df["blood_pressure"]=df["blood_pressure"].fillna(df.groupby('age_bins')["blood_pressure"].transform("median"))
  df["cholesterol"]=df["cholesterol"].fillna(df.groupby('age_bins')["cholesterol"].transform("median")).fillna(df["cholesterol"].median())
  df["bmi"]=df['bmi'].fillna(df.groupby('glucose_bins')["bmi"].transform("median")).fillna(df["bmi"].median())
  df["glucose_level"]=df["glucose_level"].fillna(df.groupby("cholesterol")["glucose_level"].transform('median')).fillna(df["glucose_level"].median())
  df["heart_rate"]=df['heart_rate'].fillna(df.groupby('age_bins')["heart_rate"].transform('median')).fillna(df["heart_rate"].median())

  df.drop(columns=["age_bins","glucose_bins","patient_id"],axis=1,inplace=True)

  if "risk_class" not in df.columns:
    return df

  else:
    df["risk_class"] = df["risk_class"].astype(str)
    df.loc[df["risk_class"].str.contains("low",case=False,na=False),'risk_class']="low_risk"
    df.loc[df["risk_class"].str.contains("medium",case=False,na=False),'risk_class']="medium_risk"
    df.loc[df["risk_class"].str.contains("high",case=False,na=False),'risk_class']="high_risk"

    df["risk_class"]=df["risk_class"].map({"low_risk":0,"medium_risk":1,"high_risk":2})

    X=df.drop(columns=["risk_class"],axis=1)
    y=df["risk_class"]

    return df,X,y


In [46]:
def train_test(X,y) -> pd.DataFrame:
  X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
  print("shape",X_train.shape,X_test.shape,y_train.shape,y_test.shape)
  return X_train,X_test,y_train,y_test


In [47]:
df=read_file("/content/patient_risk_raw.csv")
df,X,y=preprocessing(df)

X_train,X_test,y_train,y_test=train_test(X,y)

gnb=GaussianNB()
gnb.fit(X_train,y_train)
y_pred=gnb.predict(X_test)

missing value in dataset
 patient_id         0
age               30
blood_pressure    48
cholesterol       40
bmi               32
glucose_level     28
heart_rate        43
risk_class         0
dtype: int64
sample of data
   patient_id   age  blood_pressure  cholesterol   bmi  glucose_level  \
0      P1000  35.0           100.7        192.6  23.8           83.2   
1      P1001  28.0           105.5        181.2  23.2           84.8   
2      P1002  53.0           116.4        219.3  26.9           90.6   
3      P1003  51.0           121.9        193.4  28.6          115.0   
4      P1004  39.0           104.2        172.8  20.5           80.1   

   heart_rate   risk_class  
0         NaN     Low_Risk  
1        71.0     Low_Risk  
2        86.0  Medium Risk  
3        88.0  Medium_Risk  
4        62.0     Low_Risk  
shape (488, 6) (122, 6) (488,) (122,)


/tmp/ipykernel_563/330530522.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["blood_pressure"]=df["blood_pressure"].fillna(df.groupby('age_bins')["blood_pressure"].transform("median"))
/tmp/ipykernel_563/330530522.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["cholesterol"]=df["cholesterol"].fillna(df.groupby('age_bins')["cholesterol"].transform("median")).fillna(df["cholesterol"].median())
/tmp/ipykernel_563/330530522.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or obs

In [48]:
print(accuracy_score(y_test, y_pred))
print(precision_score(y_test, y_pred, average="macro"))
print(recall_score(y_test, y_pred, average="macro"))
print(f1_score(y_test, y_pred, average="macro"))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


0.9262295081967213
0.9491525423728814
0.9189886480908153
0.9285596289984005
[[32  2  0]
 [ 0 50  0]
 [ 0  7 31]]
              precision    recall  f1-score   support

           0       1.00      0.94      0.97        34
           1       0.85      1.00      0.92        50
           2       1.00      0.82      0.90        38

    accuracy                           0.93       122
   macro avg       0.95      0.92      0.93       122
weighted avg       0.94      0.93      0.93       122



In [49]:
from sklearn.model_selection import cross_val_score
cross_val=cross_val_score(gnb,X_train,y_train,cv=5)
print(cross_val)
print(cross_val.mean())

[0.8877551  0.85714286 0.79591837 0.87628866 0.89690722]
0.8628024405638544


In [50]:
# 100 test data from claude

original_test_data=read_file("/content/patient_risk_TEST_100.csv")
original_result=read_file("/content/patient_risk_TEST_100_ANSWERS.csv")

original_result["true_risk_class"]=original_result["true_risk_class"].map({
    "Low_Risk":0,
    "Medium_Risk":1,
    "High_Risk":2
})

test_actual=original_result["true_risk_class"]

test_data=preprocessing(original_test_data)
test_pred=gnb.predict(test_data)

print(accuracy_score(test_actual, test_pred))
print(precision_score(test_actual, test_pred, average="macro"))
print(recall_score(test_actual, test_pred, average="macro"))
print(f1_score(test_actual, test_pred, average="macro"))
print(confusion_matrix(test_actual, test_pred))
print(classification_report(test_actual, test_pred))


missing value in dataset
 patient_id        0
age               0
blood_pressure    0
cholesterol       0
bmi               0
glucose_level     0
heart_rate        0
dtype: int64
sample of data
   patient_id   age  blood_pressure  cholesterol   bmi  glucose_level  \
0      T2000  81.0           152.0        278.8  26.9          143.3   
1      T2001  32.0           118.2        153.7  19.3           88.0   
2      T2002  40.0            94.2        180.7  27.2           84.8   
3      T2003  52.0           111.4        218.5  22.2          104.9   
4      T2004  55.0           156.0        275.3  33.1          115.3   

   heart_rate  
0        97.0  
1        68.0  
2        70.0  
3        87.0  
4        85.0  
missing value in dataset
 patient_id         0
true_risk_class    0
dtype: int64
sample of data
   patient_id true_risk_class
0      T2000       High_Risk
1      T2001        Low_Risk
2      T2002        Low_Risk
3      T2003     Medium_Risk
4      T2004       High_Risk
0.91


/tmp/ipykernel_563/330530522.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["blood_pressure"]=df["blood_pressure"].fillna(df.groupby('age_bins')["blood_pressure"].transform("median"))
/tmp/ipykernel_563/330530522.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["cholesterol"]=df["cholesterol"].fillna(df.groupby('age_bins')["cholesterol"].transform("median")).fillna(df["cholesterol"].median())
/tmp/ipykernel_563/330530522.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or obs

# after scalling tha data


In [51]:
main = pd.read_csv("/content/patient_risk_raw.csv")
main,X,y=preprocessing(main)

X_train,X_test,y_train,y_test=train_test(X,y)
scaler=StandardScaler()

X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

gnb.fit(X_train,y_train)
pred1=gnb.predict(X_test)

print(accuracy_score(y_test, pred1))
print(precision_score(y_test, pred1, average="macro"))
print(recall_score(y_test, pred1, average="macro"))
print(f1_score(y_test, pred1, average="macro"))
print(confusion_matrix(y_test, pred1))
print(classification_report(y_test, pred1))


shape (488, 6) (122, 6) (488,) (122,)
0.9262295081967213
0.9491525423728814
0.9189886480908153
0.9285596289984005
[[32  2  0]
 [ 0 50  0]
 [ 0  7 31]]
              precision    recall  f1-score   support

           0       1.00      0.94      0.97        34
           1       0.85      1.00      0.92        50
           2       1.00      0.82      0.90        38

    accuracy                           0.93       122
   macro avg       0.95      0.92      0.93       122
weighted avg       0.94      0.93      0.93       122



/tmp/ipykernel_563/330530522.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["blood_pressure"]=df["blood_pressure"].fillna(df.groupby('age_bins')["blood_pressure"].transform("median"))
/tmp/ipykernel_563/330530522.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["cholesterol"]=df["cholesterol"].fillna(df.groupby('age_bins')["cholesterol"].transform("median")).fillna(df["cholesterol"].median())
/tmp/ipykernel_563/330530522.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or obs